# Hinglish Toxic Comment Detector — Training Notebook
**What this does:** Fine-tunes Phi-3.5-mini using QLoRA on Hinglish toxic comment data.

**Requirements:** Run this on a GPU runtime (Colab T4 free tier works).

---

## Step 1: Install Libraries

In [1]:
!pip install -q unsloth peft transformers trl datasets bitsandbytes accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.8/57.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 MB 11.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 135.7 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 869.6/869.6 kB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 127.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 134.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 82.3 MB/s eta 0:00:00:00:0

## Step 2: Check GPU

In [3]:
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU found! Go to Runtime → Change runtime type → GPU")

GPU Available: True
GPU Name: Tesla T4


AttributeError: 'torch._C._CudaDeviceProperties' object has no attribute 'total_mem'

## Step 3: Load Model (4-bit Quantized)

In [4]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Phi-3.5-mini-instruct",
    max_seq_length=1024,
    load_in_4bit=True,
)
print("Model loaded successfully!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.7: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/140 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

Model loaded successfully!


## Step 4: Add LoRA Adapters

In [5]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0,  # Unsloth optimization ke liye isse 0 kiya hai
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

trainable, total = model.get_nb_trainable_parameters()
print(f"Trainable params: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")

Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2026.5.7 patched 32 layers with 32 QKV layers, 32 O layers and 0 MLP layers.


Trainable params: 12,582,912 / 3,833,662,464 (0.33%)


## Step 5a: Upload Dataset to Colab
Run this cell first — it will ask you to select `hinglish_toxic_clean.csv` from your local machine.

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
import pandas as pd
from datasets import Dataset

# Dataset load aur check karna
df = pd.read_csv("/content/drive/MyDrive/hinglish_toxic_clean.csv")
print(f"Dataset: {len(df)} rows")
print(df["label"].value_counts())

# Prompt formatting function
def format_prompt(row):
    return {
        "text": f"""### Instruction:
Classify this Hinglish comment as 'offensive' or 'not_offensive'.

### Input:
{row['text']}

### Response:
{row['label']}"""
    }

# Dataset conversion aur splitting
dataset = Dataset.from_pandas(df)
dataset = dataset.map(format_prompt)
dataset = dataset.train_test_split(test_size=0.1, seed=42)

# Final verification
print(f"\nTrain: {len(dataset['train'])} | Test: {len(dataset['test'])}")
print(f"\nSample prompt:\n{dataset['train'][0]['text']}")

Dataset: 30452 rows
label
not_offensive    16181
offensive        14271
Name: count, dtype: int64


Map:   0%|          | 0/30452 [00:00<?, ? examples/s]


Train: 27406 | Test: 3046

Sample prompt:
### Instruction:
Classify this Hinglish comment as 'offensive' or 'not_offensive'.

### Input:
sex male race black e d oh b height female weight eye brown hair hair black in custody for misdemeanor assault count 5 of criminal trespass deg this dude also looks out like not an original old version replica of cornelius from planet of the apes

### Response:
offensive


## Step 6: Train the Model
This will take ~30-60 minutes on a T4 GPU.

In [8]:
import os
os.environ["UNSLOTH_USE_FUSED_CE_LOSS"] = "0"
import torch._dynamo
torch._dynamo.config.suppress_errors = True
print("Fixes applied!")

Fixes applied!


In [10]:
from transformers import TrainingArguments
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    dataset_text_field="text",
    max_seq_length=1024,  # Isko 1024 kiya taaki Step 3 wale model se size match ho jaye
    packing=False,
    args=TrainingArguments(
        output_dir="./results",
        num_train_epochs=1,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=50,
        eval_strategy="epoch",
        save_strategy="epoch",
        warmup_steps=100,
        weight_decay=0.01,
        report_to="none",
        torch_compile=False,
    ),
)

print("Starting training...")
trainer.train()
print("Training complete!")

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/27406 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/3046 [00:00<?, ? examples/s]

Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 27,406 | Num Epochs = 1 | Total steps = 857
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 8 x 1) = 32
 "-____-"     Trainable parameters = 12,582,912 of 3,833,662,464 (0.33% trained)


Epoch,Training Loss,Validation Loss
1,1.427365,1.410166


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

Training complete!


## Step 7: Quick Test — Try the Model

In [11]:
FastLanguageModel.for_inference(model)

test_comments = [
    "Bhai tu toh kamaal ka insaan hai",
    "Abe saale gadhe kahika nikal yahan se",
    "Yaar ye movie bohot achhi thi must watch",
    "Tujhe toh jootey maarne chahiye bewakoof",
]

for comment in test_comments:
    prompt = f"""### Instruction:
Classify this Hinglish comment as 'offensive' or 'not_offensive'.

### Input:
{comment}

### Response:
"""
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=10, temperature=0.1)
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    prediction = result.split("### Response:")[-1].strip()
    print(f"Comment: {comment}")
    print(f"Prediction: {prediction}\n")

Both `max_new_tokens` (=10) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12

Comment: Bhai tu toh kamaal ka insaan hai
Prediction: not_offensive

### Input:



Both `max_new_tokens` (=10) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Comment: Abe saale gadhe kahika nikal yahan se
Prediction: not_offensive

### Instruction



Both `max_new_tokens` (=10) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Comment: Yaar ye movie bohot achhi thi must watch
Prediction: not_offensive

### Instruction

Comment: Tujhe toh jootey maarne chahiye bewakoof
Prediction: not_offensive

### Instruction



## Step 8: Save & Download Model

This saves only the LoRA adapter files (small ~50-100MB), not the full model.

In [12]:
model.save_pretrained("toxic-detector-lora")
tokenizer.save_pretrained("toxic-detector-lora")
print("Model saved!")

import os
for f in os.listdir("toxic-detector-lora"):
    size = os.path.getsize(f"toxic-detector-lora/{f}") / 1e6
    print(f"  {f} — {size:.1f} MB")

Unsloth: Restored added_tokens_decoder metadata in toxic-detector-lora/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in toxic-detector-lora.


Model saved!
  tokenizer_config.json — 0.0 MB
  adapter_model.safetensors — 50.4 MB
  adapter_config.json — 0.0 MB
  tokenizer.model — 0.5 MB
  README.md — 0.0 MB
  chat_template.jinja — 0.0 MB
  tokenizer.json — 3.6 MB


## Step 9: Download Model Files

**Option A** — If using VS Code Colab extension, the files are already on Colab's remote disk. Download them manually from the file browser.

**Option B** — If using browser Colab, run the cell below to zip and download.

In [13]:
!zip -r toxic-detector-lora.zip toxic-detector-lora/
from google.colab import files
files.download("toxic-detector-lora.zip")
print("Download started! Check your Downloads folder.")

  adding: toxic-detector-lora/ (stored 0%)
  adding: toxic-detector-lora/tokenizer_config.json (deflated 86%)
  adding: toxic-detector-lora/adapter_model.safetensors (deflated 7%)
  adding: toxic-detector-lora/adapter_config.json (deflated 58%)
  adding: toxic-detector-lora/tokenizer.model (deflated 55%)
  adding: toxic-detector-lora/README.md (deflated 65%)
  adding: toxic-detector-lora/chat_template.jinja (deflated 61%)
  adding: toxic-detector-lora/tokenizer.json (deflated 85%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started! Check your Downloads folder.


In [ ]:
# Step 1: Install
!pip install -q unsloth peft transformers huggingface_hub

# Step 2: Load model + adapter (same as training)
from unsloth import FastLanguageModel
from peft import PeftModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Phi-3.5-mini-instruct",
    max_seq_length=1024,  # Isko 1024 kiya kyunki training isi length par hui hai
    load_in_4bit=True,
)

model = PeftModel.from_pretrained(model, "/content/drive/MyDrive/toxic-detector-lora")

# Step 3: Merge adapter into base model
merged_model = model.merge_and_unload()

# Step 4: Save merged model
merged_model.save_pretrained("merged-model")
tokenizer.save_pretrained("merged-model")
print("Merged model saved!")

# Step 5: Upload to HF Hub
from huggingface_hub import HfApi

api = HfApi()
api.upload_folder(
    folder_path="merged-model",
    repo_id="armaan-vala/hinglish-toxic-merged",
    repo_type="model",
    token="YOUR_HF_TOKEN_HERE",
)
print("Uploaded to HF Hub!")